In [23]:
%load_ext autoreload
%autoreload 2

LANGAUGE = "ts"
FULL_REPO = "pancakeswap/pancake-frontend"
REPO = "pancake-frontend"
AI_COMMITS_UNCLEAN = f"csv/{LANGAUGE}/{REPO}_AI_commits_unclean.csv"
ALL_COMMITS_UNCLEAN = f"csv/{LANGAUGE}/{REPO}_all_commits_unclean.csv"
MEREGED = f"csv/{LANGAUGE}/{REPO}_MEREGED.csv"

START_DATE = "2023-04-07"
END_DATE = "2024-01-09"



WITHIN_TIMEFRAME = f"csv/{LANGAUGE}/{REPO}_within_time_frame.csv"
TIME_FRAME_AI_EXCLUDED = f"csv/{LANGAUGE}/{REPO}_AI_removed_in_timeframe.csv"





The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [24]:
import pandas as pd
from pydriller import Repository
import os

unwanted_colums_in_original = ["branch", "committer", "author", "mention_language", "github_link", "mention"]

df = pd.read_csv("commit_mentions.csv")
df.drop(columns=unwanted_colums_in_original, inplace=True)
print(df.columns)


Index(['programming_language', 'repo_name', 'commit_id', 'tool'], dtype='str')


In [25]:
# Get the repos we'd like
print(f"Length before cleaning the data: {len(df)}")


df = df[df["repo_name"].isin([FULL_REPO])]

print(f"Length after cleaning the data: {len(df)}")
if not os.path.isdir(f"csv/{LANGAUGE}/"):
    os.mkdir(f"csv/{LANGAUGE}")

df.to_csv(AI_COMMITS_UNCLEAN, index=False)


Length before cleaning the data: 1572
Length after cleaning the data: 1003


In [26]:
data = []

for commit in Repository("https://github.com/pancakeswap/pancake-frontend").traverse_commits():
    data.append({
        "hash": commit.hash,
        "date": commit.author_date,
        "files": commit.files,
        "deletions": commit.deletions,
        "insertions": commit.insertions,
        "lines": commit.lines,

    })

df = pd.DataFrame(data)


df.to_csv(ALL_COMMITS_UNCLEAN, index=False)

In [27]:
 # To get timeframe, update on top!
df1 = pd.read_csv(AI_COMMITS_UNCLEAN)   # contains commit_id
df2 = pd.read_csv(ALL_COMMITS_UNCLEAN)   # contains hash

merged = df1.merge(df2, left_on="commit_id", right_on="hash", how="inner")

merged.to_csv(MEREGED, index=False)
df = pd.read_csv(MEREGED)

df["date"] = pd.to_datetime(df["date"], utc=True)



df = df[(df["date"] >= START_DATE) & (df["date"] <= END_DATE)]


merged.to_csv(WITHIN_TIMEFRAME, index=False)


print("Min date:", df["date"].min())
print("Max date:", df["date"].max())

Min date: 2023-04-07 05:00:33+00:00
Max date: 2024-01-03 13:12:48+00:00


In [28]:
df1 = pd.read_csv(AI_COMMITS_UNCLEAN)   # contains commit_id
df2 = pd.read_csv(ALL_COMMITS_UNCLEAN)   # contains hash
df2 = df2[(df2["date"] >= START_DATE) & (df2["date"] <= END_DATE)]
merged = df2.merge(df1, left_on="hash", right_on="commit_id", how="left", indicator=True)

df2_unmatched = merged[merged["_merge"] == "left_only"]

print(len(df2_unmatched))
df2_unmatched.to_csv(TIME_FRAME_AI_EXCLUDED, index=False)



1013
